In [2]:
import itertools
import pandas as pd
import re

# =====================================
# 1. Generate monometallic compounds
# =====================================
metals_all = [
    'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Zr', 'Nb', 'Mo', 'Ru', 'Pd', 'Ag', 'Cd',
    'Ta', 'W', 'Ir', 'Pt', 'La', 'Ce',
    'Al', 'In', 'Sn'
]

nitride_templates = ['MN', 'M2N', 'M3N2']
sulfide_templates = ['MS', 'MS2', 'M2S3']
oxide_templates = ['MO', 'M2O3', 'MO2', 'M3O4']

single_metal_compounds = []

for metal in metals_all:
    for template in nitride_templates:
        single_metal_compounds.append(
            template.replace('M', metal)
        )

    for template in sulfide_templates:
        single_metal_compounds.append(
            template.replace('M', metal)
        )

    for template in oxide_templates:
        single_metal_compounds.append(
            template.replace('M', metal)
        )

print(
    f"Number of monometallic compounds: "
    f"{len(single_metal_compounds)}"
)

# =====================================
# 2. Define five-metal composition settings
# =====================================
metals = [
    "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Ru", "Pd", "Ag", "Mo", "In"
]

MAIN_RATIOS = [0.5, 0.6]

TEMPLATES_05 = [
    [0.2, 0.1, 0.1, 0.1],
    [0.2, 0.15, 0.1, 0.05],
    [0.2, 0.2, 0.05, 0.05]
]

TEMPLATES_06 = [
    [0.1, 0.1, 0.1, 0.1],
    [0.15, 0.1, 0.1, 0.05],
    [0.2, 0.1, 0.05, 0.05]
]

TEMPLATE_DICT = {
    0.5: TEMPLATES_05,
    0.6: TEMPLATES_06
}

# =====================================
# 3. Define helper functions
# =====================================
def count_base_metal(formula):
    match = re.match(
        r"([A-Z][a-z]?)(\d*)",
        formula
    )

    if match:
        number = match.group(2)
        return int(number) if number else 1

    return 1


def canonical_key(metals, ratios):
    return tuple(
        sorted(zip(metals, ratios))
    )


def format_formula(
    all_metals,
    ratios,
    main_metal,
    suffix
):
    pairs = list(zip(all_metals, ratios))

    main_pair = [
        pair
        for pair in pairs
        if pair[0] == main_metal
    ]

    other_pairs = [
        pair
        for pair in pairs
        if pair[0] != main_metal
    ]

    other_pairs_sorted = sorted(other_pairs)
    final_pairs = other_pairs_sorted + main_pair

    formula = ""

    for metal, ratio in final_pairs:
        formula += f"{metal}{ratio}"

    return formula + suffix


# =====================================
# 4. Generate five-metal compounds
#    with all distinct ratio permutations
# =====================================
def generate_five_metal_compounds_permutation():
    results = []
    seen = set()

    for base in single_metal_compounds:
        base_m = count_base_metal(base)

        main_metal = re.match(
            r"([A-Z][a-z]?)",
            base
        ).group(1)

        for main_ratio in MAIN_RATIOS:
            templates = TEMPLATE_DICT[main_ratio]

            other_metals = [
                metal
                for metal in metals
                if metal != main_metal
            ]

            for four in itertools.combinations(
                other_metals, 4
            ):
                for template in templates:
                    # Generate all distinct ratio assignments
                    for perm_ratio in itertools.permutations(
                        template
                    ):
                        ratios = list(perm_ratio) + [
                            main_ratio
                        ]

                        scaled = [
                            round(ratio * base_m, 4)
                            for ratio in ratios
                        ]

                        all_metals = list(four) + [
                            main_metal
                        ]

                        # Remove duplicate assignments
                        key = (
                            base,
                            canonical_key(
                                all_metals,
                                scaled
                            )
                        )

                        if key in seen:
                            continue

                        seen.add(key)

                        # Remove the metal count from the
                        # parent formula before appending the anion
                        suffix = re.sub(
                            r"^\d+",
                            "",
                            base[len(main_metal):]
                        )

                        formula = format_formula(
                            all_metals,
                            scaled,
                            main_metal,
                            suffix
                        )

                        results.append({
                            "base": base,
                            "compound": formula
                        })

    return pd.DataFrame(results)


# =====================================
# 5. Generate and save the screening space
# =====================================
df = generate_five_metal_compounds_permutation()

df.to_csv(
    "five_metal_compounds_permutation.csv",
    index=False
)

print(
    "Number of five-metal compounds: "
    f"{len(df)}"
)

print(df.head())

Number of monometallic compounds: 250
Number of five-metal compounds: 4088700
  base                      compound
0  TiN    Co0.1Fe0.1Mn0.2Ni0.1Ti0.5N
1  TiN    Co0.1Fe0.2Mn0.1Ni0.1Ti0.5N
2  TiN    Co0.2Fe0.1Mn0.1Ni0.1Ti0.5N
3  TiN    Co0.1Fe0.1Mn0.1Ni0.2Ti0.5N
4  TiN  Co0.1Fe0.15Mn0.2Ni0.05Ti0.5N
